# Module 0.0 — Base-Model Viability

**Decision gate for the entire pipeline.** Does Gemma-3-12B PT refuse often enough on FaithEval-unanswerable to give dose-response headroom?

- ≥ 0.15: proceed with PT as planned
- [0.05, 0.15): run IT in parallel, pick cleaner signal
- < 0.05: switch to IT for entire pipeline

**Run on:** Colab Pro+ A100 (~5 GPU-hr) or Vast A100 80GB.

**Cost estimate:** ~$15 at Lambda; ~$5 at Vast; included in Colab Pro+.

**Prerequisite:** sanity_test.ipynb must pass first.

## Cell 1 — env setup

In [ ]:
# Colab block — uncomment
# from google.colab import userdata
# import os
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# !git clone https://github.com/<user>/desperation-circuit.git
# %cd desperation-circuit

# Vast/Lambda block — already cloned; just install
!bash scripts/setup_env.sh

## Cell 2 — load Gemma-3-12B PT

In [ ]:
import sys
sys.path.insert(0, '.')

from src.lib.model_load import load_gemma

model, tokenizer = load_gemma(variant='primary')
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}')

## Cell 3 — full FaithEval run (2,492 prompts)

Checkpoints every 100 prompts to `outputs/m0_viability/baseline_faitheval.csv`. Safe to interrupt and resume.

In [ ]:
from pathlib import Path
from src.faitheval_eval import run_eval, summary

out_dir = Path('outputs/m0_viability')
out_dir.mkdir(parents=True, exist_ok=True)

df = run_eval(
    model,
    tokenizer,
    limit=None,  # full 2,492
    checkpoint_path=out_dir / 'baseline_faitheval.csv',
    checkpoint_every=100,
)
print(f'done: {len(df)} prompts')
summary(df)

## Cell 4 — apply decision rule

In [ ]:
from src.faitheval_eval import refusal_rate

h_refuse = refusal_rate(df)
print(f'H_refuse_base = {h_refuse:.4f}')

if h_refuse >= 0.15:
    decision = 'proceed_pt'
    rationale = 'PT refuses often enough; pipeline has dose-response headroom'
elif h_refuse >= 0.05:
    decision = 'marginal_test_it_parallel'
    rationale = 'marginal; test IT in parallel and pick cleaner signal'
else:
    decision = 'switch_to_it'
    rationale = 'PT refuses too rarely; switch to IT and update extraction prompts for chat template'

print(f'decision: {decision}')
print(f'rationale: {rationale}')

(out_dir / 'decision.txt').write_text(
    f'H_refuse_base = {h_refuse:.4f}\n'
    f'decision = {decision}\n'
    f'rationale = {rationale}\n'
)

## Cell 5 — hand-label 100 to validate the judge

Required by the prereg (M0.25). Sample 100 outputs (50 judge-classified, 50 rule-classified), hand-label, compare to the auto-label, write disagreement rate to `outputs/m0_viability/judge_validation.csv`.

Do this BEFORE trusting any downstream module's classifier output.

In [ ]:
judge_sample = df[df['method'] == 'judge'].sample(min(50, (df['method']=='judge').sum()), random_state=42)
rule_sample = df[df['method'] == 'rule'].sample(50, random_state=42)
validation = pd.concat([judge_sample, rule_sample]).reset_index(drop=True)
validation['human_label'] = ''  # fill in manually
validation.to_csv(out_dir / 'judge_validation_template.csv', index=False)
print(f'wrote {len(validation)} rows to judge_validation_template.csv — hand-label and re-load to compute disagreement rate')

## Outputs

- `outputs/m0_viability/baseline_faitheval.csv` — every prompt with auto label
- `outputs/m0_viability/decision.txt` — H_refuse_base + which variant to use downstream
- `outputs/m0_viability/judge_validation_template.csv` — to be hand-labeled

## Next

After hand-labeling: open `m0.5_rome.ipynb` (ROME factual-recall control) and `m0.5_construct.ipynb` (Ferrando construct validity — the load-bearing gate).